# task_05 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_05/data/")


In [2]:

users = pd.read_csv(base / "users.csv")
catalog = pd.read_csv(base / "catalog.csv")
subscriptions = pd.read_csv(base / "subscriptions.csv")
views = pd.read_csv(base / "views.csv")

users["signup_date"] = pd.to_datetime(users["signup_date"])
subscriptions["sub_end"] = pd.to_datetime(subscriptions["sub_end"])
views["view_date"] = pd.to_datetime(views["view_date"])

Q2: For user U05, how many binge streaks are there with at least 3 consecutive days where the user had more than one view events on each day?

In [3]:
u05 = views[views["user_id"] == "U05"]
daily_counts = u05.groupby("view_date").size().reset_index(name="count")
qualifying = daily_counts[daily_counts["count"] >= 2].sort_values("view_date")
streaks = 0
run_len = 0
prev_date = None
for date in qualifying["view_date"]:
    if prev_date is not None and (date - prev_date).days == 1:
        run_len += 1
    else:
        if run_len >= 3:
            streaks += 1
        run_len = 1
    prev_date = date
if run_len >= 3:
    streaks += 1
q2 = str(int(streaks))
print(q2)


6


Q5: In the final 7 active subscription days, and restricting to dates with at least 50 total watch minutes from those events, which date had the highest Documentary share of watch time?

In [4]:
views = pd.read_csv(base / "views.csv", parse_dates=["view_date"])
subscriptions = pd.read_csv(base / "subscriptions.csv", parse_dates=["sub_start", "sub_end"])
catalog = pd.read_csv(base / "catalog.csv")

merged = views.merge(subscriptions, on="user_id").merge(catalog[["title_id", "genre"]], on="title_id")
active = merged[(merged["view_date"] >= merged["sub_start"]) & (merged["view_date"] <= merged["sub_end"])].copy()
final7 = active[(active["sub_end"] - active["view_date"]).dt.days.between(0, 6)].copy()

daily = final7.groupby("view_date", as_index=False)["watch_min"].sum().rename(columns={"watch_min": "total_watch"})
doc = final7.loc[final7["genre"] == "Documentary"].groupby("view_date", as_index=False)["watch_min"].sum().rename(columns={"watch_min": "doc_watch"})
share = daily.merge(doc, on="view_date", how="left").fillna({"doc_watch": 0})
share = share[share["total_watch"] >= 50].copy()
share["doc_share"] = share["doc_watch"] / share["total_watch"]

q5 = share.sort_values(["doc_share", "view_date"], ascending=[False, True]).iloc[0]["view_date"].date().isoformat()
print(q5)

2024-03-28
